In [10]:
# Import modules required for train.py
import os
import torch
import data_setup, engine, model_builder, utils

from torchvision import transforms

ModuleNotFoundError: No module named 'data_setup'

### Getting the data

In [11]:
import os
import requests
import zipfile
from pathlib import Path

# Setup path to the data folder
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

# If the image folder doesn't exist, download it and prepare it...
if image_path.is_dir():
  print(f"{image_path} directory exists.")
else:
  print(f"Did you find {image_path} directory, creating one...")
  image_path.mkdir(parents=True, exist_ok=True)

# Download the pizza, steak , sushi data
with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
  request =  requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
  print("Downloading pizza, steak, sushi data...")
  f.write(request.content)

# Unzip pizza, steak , sushi data
with zipfile.ZipFile(data_path  / "pizza_steak_sushi.zip", "r") as zip_ref:
  print("Unzipping the pizza, steak , sushi data")
  zip_ref.extractall(image_path)

# Remove the zip file
os.remove(data_path / "pizza_steak_sushi.zip")




Did you find data/pizza_steak_sushi directory, creating one...
Unzipping the pizza, steak , sushi data


In [15]:
!mkdir -p going_modular


##Create Datasets and DataLoader `(data_setup.py)`

In [16]:
%%writefile going_modular/data_setup.py
"""
Contains functionality for creating PyTorch DataLoaders for
image classification data.
"""

import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

NUM_WORKERS = os.cpu_count()


def create_dataloaders(
    train_dir: str,
    test_dir: str,
    transform: transforms.Compose,
    batch_size: int,
    num_workers: int = NUM_WORKERS
):
    """
    Create training and testing DataLoaders.

    Takes in a training directory and testing directory path and turns
    them into PyTorch Datasets and then into PyTorch DataLoaders.

    Args:
        train_dir (str): Path to training directory.
        test_dir (str): Path to testing directory.
        transform (torchvision.transforms.Compose):
            Transforms to perform on training and testing data.
        batch_size (int): Number of samples per batch.
        num_workers (int): Number of workers per DataLoader.

    Returns:
        tuple: (train_dataloader, test_dataloader, class_names)
            - train_dataloader (DataLoader): DataLoader for training set.
            - test_dataloader (DataLoader): DataLoader for test set.
            - class_names (list): List of class names found in training directory.
    """
    # Use ImageFolder to create dataset(s)
    train_data = datasets.ImageFolder(train_dir, transform=transform)
    test_data = datasets.ImageFolder(test_dir, transform=transform)

    # Get class names
    class_names = train_data.classes

    # Turn images into DataLoaders
    train_dataloader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )

    test_dataloader = DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_dataloader, test_dataloader, class_names


Writing going_modular/data_setup.py


If we'd like to make DataLoader's we can now use the function within data_setup.py like so:

In [17]:
# Import data_setup.py
from going_modular import data_setup

# Create train/test dataloader and get class names as a list
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(...)


TypeError: create_dataloaders() missing 3 required positional arguments: 'test_dir', 'transform', and 'batch_size'

##Making a model `(model_builder.py)`

Here we put the model into a file so that we can reuse it again and again ad infinitum
 Thus we put  our TinyVGG() model class into a script with the line %%writefile going_modular/model_builder.py:

In [18]:
%%writefile pyTorch model code to instantiate a TinyVGG model.
"""
contains the TinyVGG architecture from the CNN explainer in PyTorch
"""

import torch
from torch import nn

class TinyVGG(nn.Module):
  """Replicates the TinyVGG architecture. from the CNN explainer website in PyTorch.
  Args:
    input_shape: An integer indicating number of input channels.
    hidden_units: An integer indicating number of hidden units between layers.
    output_shape: An integer indicating number of output units.
  """
  def __init__(self, input_shape: int, hidden_units: int, output_shape: int) -> None:
    super().__init__()
    self.conv_block_1 = nn.Sequential(
      nn..Coinv2d(in_channels[input_shape,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
      nn.ReLU(),
      nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1m
                  padding=0)
       nn.ReLU(),
       nn.MaxPool2d(kernel_size=2,
                    stride=2)

    )
    self.conv_block_2 = nn.Sequential(
      nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
      nn.ReLU(),
      nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
      nn.ReLU(),
      nn.MaxPool2d(2d)
    )

    self.classifier = nn.Sequential(
      nn.Flatten(),
      #Where did this in_fearure shape comes from?
      # It's bacause each layer of our network compresses and changes the shape of our inputs data.
      nn.Linear(in_fearure=hidden_units*13*13,
      out_fearures=output_shape)

    )

  def forward(self, x: torch.Tensor):
  x = self.conv_block_1(x)
  x = self.conv_block_2(x)
  x = self.classifier(x)
  return x
# return self.classifier(self.conv_block_2(self.conv_block_1(x))) # <- leverage the benefits of operator fusion


UsageError: unrecognized arguments: model code to instantiate a TinyVGG model.


Now instead of coding the TinyVGG model from scratch every time , we can import it using:

In [ ]:
import torch
# Import the model_builder.py
from going_modular import model_builder

# Instantiate an instance of the model from the "model_builder.py" script
torch.manual_seed(42)
model = model_builder.TinyVGG(input_shaoe=3,
                              hidden_units=10,
                              output_shape=len(class_names)).to(device)

##Creating `train_step()` and `test_step()` functions and `train()` to combine them.

1 `train_step()` - takes in a model, a `DataLoader`, a loss function and an optimizer and trains the model on the DataLoader.
2 `test_step()` takes in a model a `DataLoader` and a loss function and evaluate the model in the `DataLoader`
3 `train()` - performs 1. and 2. together for a given number of epochs and  returns a results dictionary.
Since these will be the engine of our model training, we can put them all into a Python script called engine.py with the line %%writefile going_modular/engine.py:

In [19]:
%%writefile going_modular/engine.py
"""
Contains functions for training and testing a PyTorch model.
"""

import torch
from tqdm.auto import tqdm
from typing import Dict, List, Tuple


def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device) -> Tuple[float, float]:
    """
    Trains a PyTorch model for a single epoch.
    Performs forward pass, loss calculation, and optimizer step.

    Returns:
        A tuple of (train_loss, train_accuracy)
        Example: (0.1112, 0.8743)
    """
    model.train()
    train_loss, train_acc = 0, 0

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # 1. Forward pass
        y_pred = model(X)

        # 2. Loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Backward pass
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

        # 6. Calculate accuracy
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y_pred)

    # Average loss and accuracy
    train_loss /= len(dataloader)
    train_acc /= len(dataloader)

    return train_loss, train_acc


def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device) -> Tuple[float, float]:
    """
    Tests a PyTorch model for a single epoch.

    Returns:
        A tuple of (test_loss, test_accuracy)
        Example: (0.0223, 0.8985)
    """
    model.eval()
    test_loss, test_acc = 0, 0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)

            # 1. Forward pass
            test_pred_logits = model(X)

            # 2. Loss
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            # 3. Accuracy
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += (test_pred_labels == y).sum().item() / len(test_pred_labels)

    test_loss /= len(dataloader)
    test_acc /= len(dataloader)

    return test_loss, test_acc


def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device) -> Dict[str, List]:
    """
    Trains and tests a PyTorch model for a given number of epochs.

    Returns:
        A dictionary containing training and testing loss/accuracy per epoch.
    """
    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []}

    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           device=device)

        test_loss, test_acc = test_step(model=model,
                                        dataloader=test_dataloader,
                                        loss_fn=loss_fn,
                                        device=device)

        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f}"
        )

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

    return results


Writing going_modular/engine.py


In [20]:
# Import engine.py
from going_modular import engine

# use train() by calling it from engine.py
engine.train()

TypeError: train() missing 7 required positional arguments: 'model', 'train_dataloader', 'test_dataloader', 'optimizer', 'loss_fn', 'epochs', and 'device'

##Creating a function to save the model `(utils.py)`
Let's save our save_model() function to a file called utils.py with the line %%writefile going_modular/utils.py:

In [21]:
%%writefile going_modular/utils.py
"""contains various utility functions for PyTorch model training and saving."""
import torch
 from pathlib import Path

 def save_model(model: torch.nn.Module,
                target_dir: str,
                model_name: str):
   """ Saves a PyTorch model to a target directory.
   Args:
       model: A target PyTorch model to save.
       target_dir: A directory for saving the model to.
       model_name: A filename for the saved model.
       Should include either ".pth" or ".pt" as the file extension.

       Example usage:
       save_model(model=model_0,
                  target_dir="models",
                  model_name="05_going_modular_tingvgg_model.pth")
   """
  #  Create target directory
  target_dir_path = Path(target_dir)
  target_dir_path.mkdir(parents=True,
                        exist_on=True)

  # Create model save path
  assert model_name.endswith(".pth") or model_name.endawith(".pt"), "model_name should end with '.pt' or '.pth'"
  model_save_path = target_dir_path / model_name

  # Save the model stae_dict()
  print(f"[INFO] Saving model to: {model_save_path}")
  torch.save(obj=model.state_dict(),
             f=model_save_path)





Writing going_modular/utils.py


Now if we wanted to use our save_model() function, instead of writing it all over again, we can import it and use it via:

In [22]:
# Import utils.py
from going_modular import utils

# Save a model to file
save_model(model=...
           target_dir=...,
           model_name=...)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (ipython-input-1904613542.py, line 5)

 Train, evaluate and save the model `(train.py)`

```
As previously discussed, you'll often come across PyTorch repositories that combine all of their functionality together in a train.py file.

This file is essentially saying "train the model using whatever data is available".

In our train.py file, we'll combine all of the functionality of the other Python scripts we've created and use it to train a model.

This way we can train a PyTorch model using a single line of code on the command line:

python train.py

```



To create train.py we'll go through the following steps:

    Import the various dependencies, namely torch, os, torchvision.transforms and all of the scripts from the going_modular directory, data_setup, engine, model_builder, utils.

    Note: Since train.py will be inside the going_modular directory, we can import the other modules via import ... rather than from going_modular import ....

    Setup various hyperparameters such as batch size, number of epochs, learning rate and number of hidden units (these could be set in the future via Python's argparse).
    Setup the training and test directories.
    Setup device-agnostic code.
    Create the necessary data transforms.
    Create the DataLoaders using data_setup.py.
    Create the model using model_builder.py.
    Setup the loss function and optimizer.
    Train the model using engine.py.
    Save the model using utils.py.


In [23]:
%%writefile going_modular/train.py

""" Trains a PyTorch image classification model using device-agnostic code."""

import os
import torch
import data_setup, model_builder, utils

from torchvision import transforms

# Setup hyperparameters
NUM_EPOCHS = 5
BATCH_SIZE = 32
HIDDEN_UNITS = 10
LEARNING_RATE = 0.001

# Setup directories
train_acc = "data/pizza_steak_sushi/train"
test_dir = "data/pizza_steak_sushi/test"

# Setup target deivice
device = "cuda" if torch.cuda.is_available() else "cpu"

# Create transforms
data_transform = transforms.Compose([
  transforms.Resize((64, 64)),
  transforms.ToTensor()
])

# Create DataLoaders with help from data_setup.py
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=data_transform,
    batch_size=BATCH_SIZE
)

# Create model with help from model_builder.py
model = model_builder.TinyVGG(
    input_shape=3,
    hidden_units=HIDDEN_UNITS,
    output_shape=len(class_names)
).to(device)


# Set loss and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),
                             lr=LEARNING_RATE)


# Start training with help from engine.py
engine.train(model=model,
             train_dataloader=train_dataloader,
             test_dataloader=test_dataloader,
             loss_fn=loss_fn,
             optimizer=optimizer,
             epochs=NUM_EPOCHS,
             device=device)

# Save the model with help from utils.py
utils.save_model(model=model,
                 target_dir="models",
                 model_name="05_going_modular_script_mode_tinyvgg_model.pth")


Writing going_modular/train.py


Woohoo!

Now we can train a PyTorch model by running the following line on the command line:

`python train.py`

